
# Hospital Readmission Prediction — AutoGluon Model Training

Dataset: cleaned `diabetic_data_cleaned.csv` from EDA pipeline  
Target: `readmitted` — `NO` / `>30` / `<30`

**Pipeline**
1. Imports & paths  
2. Load cleaned data  
2b. Feature engineering (severity score, lab/med per day, clinical flags)  
3. Stratified 5% unseen hold-out → `unseen_data.csv`  
4. 80 / 20 train / test split  
5. Null imputation — fit on train, apply to train & test only (unseen untouched)  
6. Feature selection (21 features) + inverse-frequency sample weights → save CSVs  
7. SMOTE oversampling — balance minority classes on train set  
8. AutoGluon `TabularPredictor` training (`f1_macro`, sample weights)  
9. Leaderboard  
10. Hold-out evaluation (metrics + confusion matrix)  
11. Feature importance  
12. Final evaluation on unseen data


## 1. Imports & Paths

In [ ]:

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, balanced_accuracy_score,
    confusion_matrix, f1_score
)

# ── paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path('..').resolve()
PROCESSED_DIR = ROOT / 'processed'
OUT_DIR       = ROOT / 'outputs'
(OUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)

DATA_PATH     = PROCESSED_DIR / 'diabetic_data_cleaned.csv'
MODEL_DIR     = str(OUT_DIR / 'autogluon_model')

TARGET   = 'readmitted'
FEATURES = [
    # ── Original top features ─────────────────────────────────────
    'discharge_group',        # categorical
    'admission_source_group', # categorical
    'admission_type_group',   # categorical
    'medical_specialty',      # categorical
    'diabetesMed',            # categorical
    'diag_1_cat',             # categorical
    'age',                    # categorical (range strings)
    'insulin',                # categorical
    'has_emergency',          # numeric
    'severity_score',         # numeric (replaces total_visits; weighted utilisation)
    # ── 5 additional (by EDA feature importance) ──────────────────
    'number_inpatient',       # numeric  — highest MI score
    'number_diagnoses',       # numeric  — MI score
    'time_in_hospital',       # numeric  — clinical relevance
    'change',                 # categorical — high chi² significance
    #'num_lab_procedures',     # numeric  — MI score
    # ── 5 clinical engineered features ───────────────────────────
    'lab_per_day',            # numeric  — normalised test intensity
    'med_per_day',            # numeric  — normalised medication burden
    'diabetes_med_count',     # numeric  — polypharmacy proxy
    'high_dose_insulin',      # binary   — intensive glucose management
    'is_cardio_diabetic',     # binary   — cardio-metabolic comorbidity
    'is_discharged_home',     # binary   — discharge destination
]

print('ROOT       :', ROOT)
print('DATA_PATH  :', DATA_PATH.exists(), DATA_PATH)
print('MODEL_DIR  :', MODEL_DIR)
print(f'Features   : {len(FEATURES)} total — {FEATURES}')


ROOT       : /
DATA_PATH  : True processed/diabetic_data_cleaned.csv
MODEL_DIR  : outputs/autogluon_model
Features   : 20 total — ['discharge_group', 'admission_source_group', 'admission_type_group', 'medical_specialty', 'diabetesMed', 'diag_1_cat', 'age', 'insulin', 'has_emergency', 'severity_score', 'number_inpatient', 'number_diagnoses', 'time_in_hospital', 'change', 'lab_per_day', 'med_per_day', 'diabetes_med_count', 'high_dose_insulin', 'is_cardio_diabetic', 'is_discharged_home']


## 2. Load Cleaned Data

In [6]:
df_full = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded: {df_full.shape}  ({df_full.shape[0]:,} rows × {df_full.shape[1]} columns)')
print(f'\nTarget distribution:')
print(df_full[TARGET].value_counts(normalize=True).mul(100).round(2).to_string())
df_full.head()

Loaded: (100097, 57)  (100,097 rows × 57 columns)

Target distribution:
readmitted
NO     53.14
>30    35.51
<30    11.35


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,diag_2_cat,diag_3_cat,admission_type_id_label,discharge_disposition_id_label,admission_source_id_label,total_visits,has_emergency,admission_type_group,discharge_group,admission_source_group
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,...,Unknown,Unknown,NaN,Not Mapped,Physician Referral,0,0,Unknown,Unknown,Referral
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,...,Diabetes,Other,Emergency,Discharged to home,Emergency Room,0,0,Emergency,Home,Emergency
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,...,Diabetes,Supplementary (V),Emergency,Discharged to home,Emergency Room,3,0,Emergency,Home,Emergency
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,...,Diabetes,Circulatory,Emergency,Discharged to home,Emergency Room,0,0,Emergency,Home,Emergency
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,...,Neoplasms,Diabetes,Emergency,Discharged to home,Emergency Room,0,0,Emergency,Home,Emergency



## 2b. Feature Engineering




In [ ]:

for c in ['number_outpatient', 'number_emergency', 'number_inpatient']:
    df_full[c] = pd.to_numeric(df_full[c], errors='coerce').fillna(0)

df_full['severity_score'] = (
    df_full['number_emergency'] * 3 +
    df_full['number_inpatient'] * 2 +
    df_full['number_outpatient']
)
df_full['has_emergency'] = (df_full['number_emergency'] > 0).astype(int)

# ── 2. Lab_per_Day & Med_per_Day ──────────────────────────────────────────────
t = df_full['time_in_hospital'].clip(lower=1)
df_full['lab_per_day'] = df_full['num_lab_procedures'] / t
df_full['med_per_day'] = df_full['num_medications']    / t

# ── 3. Diabetes_Med_Count & High_Dose_Insulin ─────────────────────────────────
DIABETES_MEDS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'examide', 'citoglipton',
]
present_meds = [m for m in DIABETES_MEDS if m in df_full.columns]
df_full['diabetes_med_count'] = (df_full[present_meds] != 'No').sum(axis=1)
df_full['high_dose_insulin']  = (
    (df_full['insulin'] == 'Up') & (df_full['change'] == 'Ch')
).astype(int)

# ── 4. Is_Cardio_Diabetic ─────────────────────────────────────────────────────
def _is_cardiovascular(code):
    try:
        return 390 <= float(str(code).strip()) < 460
    except (ValueError, TypeError):
        return False

df_full['is_cardio_diabetic'] = (
    df_full['diag_1'].apply(_is_cardiovascular) & (df_full['diabetesMed'] == 'Yes')
).astype(int)

# ── 5. Is_Discharged_Home ─────────────────────────────────────────────────────
df_full['is_discharged_home'] = (df_full['discharge_disposition_id'] == 1).astype(int)

print('Engineered features added:')
new_cols = ['severity_score','has_emergency','lab_per_day','med_per_day',
            'diabetes_med_count','high_dose_insulin','is_cardio_diabetic','is_discharged_home']
print(df_full[new_cols].describe().round(3).to_string())
print(f'\ndf_full shape: {df_full.shape}')


Engineered features added:
       severity_score  has_emergency  lab_per_day  med_per_day  diabetes_med_count  high_dose_insulin  is_cardio_diabetic  is_discharged_home
count      100097.000     100097.000   100097.000   100097.000          100097.000         100097.000          100097.000          100097.000
mean            2.231          0.112       14.302        5.062               1.185              0.111               0.231               0.602
std             4.579          0.315       11.939        3.807               0.922              0.314               0.422               0.490
min             0.000          0.000        0.071        0.083               0.000              0.000               0.000               0.000
25%             0.000          0.000        6.500        2.600               1.000              0.000               0.000               0.000
50%             0.000          0.000       10.833        4.000               1.000              0.000               0.000


## 3. Stratified 5% Unseen Hold-Out → `unseen_data.csv`


In [8]:

df_seen, df_unseen = train_test_split(
    df_full,
    test_size=0.05,
    stratify=df_full[TARGET],
    random_state=42
)

unseen_path = PROCESSED_DIR / 'unseen_data.csv'
df_unseen.to_csv(unseen_path, index=False)
print(f'Unseen data  : {df_unseen.shape}')
print(f'Remaining 95%: {df_seen.shape}')
print(f'\nUnseen class distribution (%):')
print(df_unseen[TARGET].value_counts(normalize=True).mul(100).round(2).to_string())


Unseen data  : (5005, 64)
Remaining 95%: (95092, 64)

Unseen class distribution (%):
readmitted
NO     53.15
>30    35.50
<30    11.35



## 4. 80 / 20 Train / Test Split


In [9]:

train_df, test_df = train_test_split(
    df_seen,
    test_size=0.20,
    stratify=df_seen[TARGET],
    random_state=42
)

train_path = PROCESSED_DIR / 'train.csv'
test_path  = PROCESSED_DIR / 'test.csv'

print(f'Training set : {train_df.shape}')
print(f'Test set     : {test_df.shape}')
for name, split in [('Train', train_df), ('Test', test_df)]:
    print(f'\n{name} class distribution (%):')
    print(split[TARGET].value_counts(normalize=True).mul(100).round(2).to_string())


Training set : (76073, 64)
Test set     : (19019, 64)

Train class distribution (%):
readmitted
NO     53.14
>30    35.51
<30    11.35

Test class distribution (%):
readmitted
NO     53.14
>30    35.51
<30    11.35


In [10]:

# Null counts before imputation (train set)
print('Null counts in train set before imputation:')
null_counts = train_df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print('  No nulls found.')
else:
    print(null_counts.to_string())


Null counts in train set before imputation:
race                               1701
payer_code                        30167
medical_specialty                 37252
diag_1                               14
diag_2                              275
diag_3                             1082
A1Cresult                         63322
admission_type_id_label            4016
discharge_disposition_id_label     2801
admission_source_id_label          5054



## 5. Null Imputation — Apply to Train & Test Only


| Column | Strategy |
|---|---|
| `A1Cresult` | `'Not Measured'` — absence itself is a clinical signal |
| `medical_specialty` | `'Missing'` |
| `race` | Mode fitted from train set |
| `diag_1` / `diag_2` / `diag_3` | `'000'` — unknown ICD code placeholder |
| `diag_1_cat` | `'Unknown'` |
| `admission_type_group`, `admission_source_group`, `discharge_group` | `'Unknown'` |


In [11]:

# ── Fit imputation values on train only ───────────────────────────────────────
race_mode = train_df['race'].mode()[0] if 'race' in train_df.columns else None

FILL_MAP = {}
if 'A1Cresult'        in train_df.columns: FILL_MAP['A1Cresult']         = 'Not Measured'
if 'medical_specialty' in train_df.columns: FILL_MAP['medical_specialty'] = 'Missing'
for col in ['diag_1', 'diag_2', 'diag_3']:
    if col in train_df.columns: FILL_MAP[col] = '000'
if 'diag_1_cat'       in train_df.columns: FILL_MAP['diag_1_cat']        = 'Unknown'
for col in ['admission_type_group', 'admission_source_group', 'discharge_group']:
    if col in train_df.columns: FILL_MAP[col] = 'Unknown'

def apply_imputation(df, fill_map, race_mode=None):
    df = df.copy()
    if race_mode is not None and 'race' in df.columns:
        df['race'] = df['race'].fillna(race_mode)
    for col, val in fill_map.items():
        df[col] = df[col].fillna(val)
    return df

# Apply only to train and test — unseen data is intentionally left as-is
train_df = apply_imputation(train_df, FILL_MAP, race_mode)
test_df  = apply_imputation(test_df,  FILL_MAP, race_mode)

print(f'Race fill value (mode from train): {race_mode!r}')
print('Constant fill values applied:')
for col, val in FILL_MAP.items():
    print(f'  {col:<35s} → {val!r}')

imputed_cols = list(FILL_MAP.keys()) + (['race'] if race_mode else [])
print('\nNull counts after imputation (imputed columns only):')
for name, split in [('Train', train_df), ('Test', test_df)]:
    remaining = split[[c for c in imputed_cols if c in split.columns]].isnull().sum()
    remaining = remaining[remaining > 0]
    if remaining.empty:
        print(f'  {name}: 0 nulls remaining ✓')
    else:
        print(f'  {name} remaining nulls:\n{remaining.to_string()}')


Race fill value (mode from train): 'Caucasian'
Constant fill values applied:
  A1Cresult                           → 'Not Measured'
  medical_specialty                   → 'Missing'
  diag_1                              → '000'
  diag_2                              → '000'
  diag_3                              → '000'
  diag_1_cat                          → 'Unknown'
  admission_type_group                → 'Unknown'
  admission_source_group              → 'Unknown'
  discharge_group                     → 'Unknown'

Null counts after imputation (imputed columns only):
  Train: 0 nulls remaining ✓
  Test: 0 nulls remaining ✓



## 6. Feature Selection


In [ ]:

# ── Validate and select features 
missing_feats = [f for f in FEATURES if f not in train_df.columns]
if missing_feats:
    print(f'WARNING — missing columns (will be skipped): {missing_feats}')
    FEATURES = [f for f in FEATURES if f in train_df.columns]

train_df  = train_df[FEATURES + [TARGET]].copy()
test_df   = test_df[FEATURES + [TARGET]].copy()

# ── Median-fill remaining numeric NaN (fit on train only) 
num_cols_sel = train_df[FEATURES].select_dtypes(include='number').columns.tolist()
median_fills = train_df[num_cols_sel].median()
for split_df in [train_df, test_df]:
    split_df[num_cols_sel] = split_df[num_cols_sel].fillna(median_fills)

# ── Carve a REAL validation set BEFORE SMOTE (15% of train) 
# This is the key fix: AutoGluon will use this as tuning_data so score_val
# reflects the true data distribution, not synthetic SMOTE samples.
train_real, val_real = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df[TARGET],
    random_state=42,
)

# ── Compute inverse-frequency sample weights on train_real only ───────────────
class_counts = train_real[TARGET].value_counts()
n_total      = len(train_real)
n_classes    = len(class_counts)
class_weight = (n_total / (n_classes * class_counts)).to_dict()
train_real = train_real.copy()
train_real['sample_weight'] = train_real[TARGET].map(class_weight)

# Save CSVs (feature-selected, imputed — before SMOTE)
train_df.to_csv(train_path,   index=False)
test_df.to_csv(test_path,    index=False)

print(f'Selected features ({len(FEATURES)}): {FEATURES}')
print(f'\nSplit sizes — train_real: {len(train_real):,}  val_real: {len(val_real):,}  test: {len(test_df):,}')
print(f'\nClass weights (inverse frequency):')
for cls, w in class_weight.items():
    print(f'  {cls:<5s}: {w:.4f}  (n={class_counts[cls]:,})')
print(f'\nRemaining nulls in train_real: {train_real[FEATURES].isnull().sum().sum()}')


Selected features (20): ['discharge_group', 'admission_source_group', 'admission_type_group', 'medical_specialty', 'diabetesMed', 'diag_1_cat', 'age', 'insulin', 'has_emergency', 'severity_score', 'number_inpatient', 'number_diagnoses', 'time_in_hospital', 'change', 'lab_per_day', 'med_per_day', 'diabetes_med_count', 'high_dose_insulin', 'is_cardio_diabetic', 'is_discharged_home']

Split sizes — train_real: 64,662  val_real: 11,411  test: 19,019

Class weights (inverse frequency):
  NO   : 0.6272  (n=34,364)
  >30  : 0.9387  (n=22,962)
  <30  : 2.9381  (n=7,336)

Remaining nulls in train_real: 0



## 7. SMOTE Oversampling (Train Set Only)


In [ ]:

import numpy as np
from imblearn.over_sampling import SMOTENC

# ── Identify categorical vs numeric features ──────────────────────────────────
X_pre = train_real[FEATURES]
# Python list of booleans — numpy bool array triggers a ValueError in current
# sklearn/_imblearn versions when used as a truth-value internally.
cat_mask     = [X_pre[col].dtype == 'object' for col in FEATURES]
cat_features = [col for col in FEATURES if X_pre[col].dtype == 'object']
num_features = [col for col in FEATURES if X_pre[col].dtype != 'object']

print(f'Categorical ({sum(cat_mask)}): {cat_features}')
print(f'Numeric     ({sum(~m for m in cat_mask)}): {num_features}')
print(f'\nClass distribution BEFORE SMOTE:')
print(train_real[TARGET].value_counts().to_string())

encoders = {}
X_enc = X_pre.copy()
for col in cat_features:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col].astype(str))
    encoders[col] = le

X_arr = X_enc.values.astype(float)
y_arr = train_real[TARGET].values

assert not np.isnan(X_arr).any(), 'NaN found before SMOTE — check imputation'

smote = SMOTENC(categorical_features=cat_mask, random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_arr, y_arr)

X_res_df = pd.DataFrame(X_res, columns=FEATURES)
for col in cat_features:
    n_cls = len(encoders[col].classes_)
    idx = X_res_df[col].round().astype(int).clip(0, n_cls - 1)
    X_res_df[col] = encoders[col].inverse_transform(idx)
for col in num_features:
    orig_dtype = X_pre[col].dtype
    X_res_df[col] = (X_res_df[col].round().astype('int64')
                     if pd.api.types.is_integer_dtype(orig_dtype)
                     else X_res_df[col].astype('float64'))

train_smote = X_res_df.copy()
train_smote[TARGET] = y_res

# Carry over sample weights 
cc = train_smote[TARGET].value_counts()
weight_s = (len(train_smote) / (len(cc) * cc)).to_dict()
train_smote['sample_weight'] = train_smote[TARGET].map(weight_s)

print(f'\nClass distribution AFTER SMOTE:')
print(train_smote[TARGET].value_counts().to_string())
print(f'\nDataset size: {len(train_real):,} → {len(train_smote):,} rows')
print(f'val_real size (real data, used as AutoGluon tuning_data): {len(val_real):,}')


Categorical (9): ['discharge_group', 'admission_source_group', 'admission_type_group', 'medical_specialty', 'diabetesMed', 'diag_1_cat', 'age', 'insulin', 'change']
Numeric     (-29): ['has_emergency', 'severity_score', 'number_inpatient', 'number_diagnoses', 'time_in_hospital', 'lab_per_day', 'med_per_day', 'diabetes_med_count', 'high_dose_insulin', 'is_cardio_diabetic', 'is_discharged_home']

Class distribution BEFORE SMOTE:
readmitted
NO     34364
>30    22962
<30     7336

Class distribution AFTER SMOTE:
readmitted
>30    34364
NO     34364
<30    34364

Dataset size: 64,662 → 103,092 rows
val_real size (real data, used as AutoGluon tuning_data): 11,411



## 8. AutoGluon Model Training

Train a `TabularPredictor` with `f1_macro` metric and inverse-frequency sample weights on the SMOTE-augmented training set.


In [ ]:

from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label=TARGET,
    path=MODEL_DIR,
    eval_metric='f1_macro',
    sample_weight='sample_weight',
    problem_type='multiclass',
).fit(
    train_data=train_smote,        # SMOTE-balanced training data
    tuning_data=val_real,          # Real (non-synthetic) validation → honest score_val
   
    presets='best_quality',
    time_limit=3600,
    verbosity=2,
)

print('\nTraining complete.')
print('Best model    :', predictor.model_best)
print('Eval metric   :', predictor.eval_metric)


Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Darwin
Platform Machine:   x86_64
Platform Version:   Darwin Kernel Version 23.1.0: Mon Oct  9 21:28:12 PDT 2023; root:xnu-10002.41.9~6/RELEASE_ARM64_T8103
CPU Count:          8
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       0.82 GB / 16.00 GB (5.1%)
Disk Space Avail:   11.90 GB / 228.27 GB (5.2%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack i

AssertionError: Learner is already fit.


## 9. Leaderboard — Model Comparison


In [ ]:
leaderboard = predictor.leaderboard(test_df, silent=True)
display(leaderboard)


## 10. Hold-Out Evaluation (Test Set)


In [ ]:
TARGET_ORDER = ['NO', '>30', '<30']

y_true = test_df[TARGET]
y_pred = predictor.predict(test_df.drop(columns=[TARGET]))

bal_acc = balanced_accuracy_score(y_true, y_pred)
f1_mac  = f1_score(y_true, y_pred, average='macro')
f1_wt   = f1_score(y_true, y_pred, average='weighted')

print(f'Balanced Accuracy : {bal_acc:.4f}')
print(f'F1 Macro          : {f1_mac:.4f}')
print(f'F1 Weighted       : {f1_wt:.4f}')
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=TARGET_ORDER))

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=TARGET_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=TARGET_ORDER, yticklabels=TARGET_ORDER, ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix — counts')

# Row-normalised (%)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=TARGET_ORDER, yticklabels=TARGET_ORDER, ax=axes[1])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix — row % (recall)')

plt.suptitle('Hold-Out Test Set Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'figures' / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


## 11. Feature Importance


In [ ]:
feat_imp = predictor.feature_importance(test_df, silent=True)
print('Feature importance (best model):')
display(feat_imp)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp['importance'].sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Feature Importance — AutoGluon best model')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig(OUT_DIR / 'figures' / 'feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


## 12. Final Evaluation on Unseen Data


In [ ]:

df_unseen_loaded = pd.read_csv(unseen_path)

for c in ['number_outpatient', 'number_emergency', 'number_inpatient']:
    df_unseen_loaded[c] = pd.to_numeric(df_unseen_loaded[c], errors='coerce').fillna(0)

df_unseen_loaded['severity_score'] = (
    df_unseen_loaded['number_emergency'] * 3 +
    df_unseen_loaded['number_inpatient'] * 2 +
    df_unseen_loaded['number_outpatient']
)
df_unseen_loaded['has_emergency'] = (df_unseen_loaded['number_emergency'] > 0).astype(int)

t_u = df_unseen_loaded['time_in_hospital'].clip(lower=1)
df_unseen_loaded['lab_per_day'] = df_unseen_loaded['num_lab_procedures'] / t_u
df_unseen_loaded['med_per_day'] = df_unseen_loaded['num_medications']    / t_u

present_meds_u = [m for m in DIABETES_MEDS if m in df_unseen_loaded.columns]
df_unseen_loaded['diabetes_med_count'] = (df_unseen_loaded[present_meds_u] != 'No').sum(axis=1)
df_unseen_loaded['high_dose_insulin']  = (
    (df_unseen_loaded['insulin'] == 'Up') & (df_unseen_loaded['change'] == 'Ch')
).astype(int)

df_unseen_loaded['is_cardio_diabetic'] = (
    df_unseen_loaded['diag_1'].apply(_is_cardiovascular) &
    (df_unseen_loaded['diabetesMed'] == 'Yes')
).astype(int)
df_unseen_loaded['is_discharged_home'] = (
    df_unseen_loaded['discharge_disposition_id'] == 1
).astype(int)

y_unseen_true = df_unseen_loaded[TARGET]
y_unseen_pred = predictor.predict(df_unseen_loaded.drop(columns=[TARGET]))

bal_acc_u = balanced_accuracy_score(y_unseen_true, y_unseen_pred)
f1_mac_u  = f1_score(y_unseen_true, y_unseen_pred, average='macro')

print(f'Unseen — Balanced Accuracy : {bal_acc_u:.4f}')
print(f'Unseen — F1 Macro          : {f1_mac_u:.4f}')
print('\nClassification Report (unseen):')
print(classification_report(y_unseen_true, y_unseen_pred, target_names=TARGET_ORDER))

cm_u = confusion_matrix(y_unseen_true, y_unseen_pred, labels=TARGET_ORDER)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm_u, annot=True, fmt='d', cmap='Purples',
            xticklabels=TARGET_ORDER, yticklabels=TARGET_ORDER, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Unseen Data')
plt.tight_layout()
plt.savefig(OUT_DIR / 'figures' / 'confusion_matrix_unseen.png', dpi=120, bbox_inches='tight')
plt.show()
